# SecureFinger — upgrades: FVC multi-session evaluation + OPA optimisation
Runtime → **T4 GPU**. Results go to Google Drive (`MyDrive/securefinger_results_upgrades`), so they survive a disconnect.

## 1. Upload the NEW code (`securefinger.zip`) and install

In [ ]:
from google.colab import files
files.upload()  # choose securefinger.zip
!unzip -q -o securefinger.zip
%cd /content/securefinger
!pip -q install tenseal fastapi httpx cryptography opencv-python-headless pyjwt scikit-learn

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/securefinger_results_upgrades'
WEIGHTS = '/content/drive/MyDrive/securefinger_results_ext/encoder_seed0.pt'  # trained in the previous notebook
!mkdir -p "$OUT"
!ls -la "$WEIGHTS"

## 3. Get SOCOFing (needed for the optimisation)
Use **ONE** option.

In [ ]:
# Option A: Kaggle API token
import os
os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_TOKEN_HERE'
!pip -q install -U kaggle
!kaggle datasets download -d ruizgara/socofing -p /content
!unzip -q -o /content/socofing.zip -d /content/socofing_raw

In [ ]:
# Option B: dataset zip already in Google Drive
!unzip -q -o "/content/drive/MyDrive/archive.zip" -d /content/socofing_raw

In [ ]:
DATA = '/content/socofing_raw/SOCOFing'
!ls $DATA

## 4. FVC multi-session evaluation (about 10–15 minutes)
Downloads FVC2000/2002/2004 set B and the Neurotechnology sample databases (all free), evaluates the SOCOFing encoder zero-shot, then fine-tunes with 2-fold finger-disjoint cross-validation (3 seeds). Check that the database list shows 8 impressions per finger.

In [ ]:
!python experiments/fvc_eval.py --root /content/fvc --weights "$WEIGHTS" --out "$OUT/fvc" 2>&1 | tee "$OUT/fvc_log.txt"

## 5. OPA optimisation on the hard protocol (about 90 minutes)
25 OPA evaluations + 25 random-search evaluations (5 epochs each), then default vs OPA-best retrained with 3 seeds for 15 epochs and tested on the untouched test split. Keep the tab open (run `caffeinate -d` on the Mac).

In [ ]:
!python experiments/optimize.py --data $DATA --out "$OUT/opt" --budget 25 --pop 5 --epochs 5 --final-epochs 15 2>&1 | tee "$OUT/opt_log.txt"

## 6. Summary to paste back

In [ ]:
import json
f = json.load(open(f'{OUT}/fvc/fvc.json'))
print('FVC databases:', {k: v['fingers'] for k, v in f['databases'].items()})
print('zero-shot:'); [print(f'  {k:16s} EER {v["eer"]*100:6.2f}%  FMR100 {v["fnmr_at_fmr_1e-2"]*100:6.2f}%') for k, v in f['zero_shot'].items()]
print('fine-tuned (pooled real):', {k: f"{v['mean']*100:.2f} ± {v['std']*100:.2f}%" for k, v in f['fine_tuned']['summary'].items()})
try:
    o = json.load(open(f'{OUT}/opt/optimization.json'))
    print('default short-budget val EER:', o['default_val_eer_hard'])
    print('OPA best:', o['opa'])
    print('random best:', o.get('random'))
    for k, v in o.get('final', {}).items():
        print('==', k, v['hp']); [print(f'   {m:40s} {s["mean"]*100:.4f} ± {s["std"]*100:.4f}') for m, s in v['summary'].items()]
except FileNotFoundError:
    print('optimisation not finished yet')

## 7. Download everything

In [ ]:
!cd "$OUT" && zip -q -r /content/results_upgrades.zip .
from google.colab import files
files.download('/content/results_upgrades.zip')